# B4 jeepney RL baseline

Traffic-biased quadrilateral baselines on the physical drivable street network.
The minimum-area rule is inspired by polygonal morphology bounds discussed in
https://arxiv.org/html/2603.28385v1.


## 1. Traffic-Biased Quadrilateral Baseline Generator

Builds traffic-weighted quadrilateral routes on the physical drive graph, then stitches the shortest physical path between the four selected anchors.

In [1]:
import pandas as pd
import secrets
from pathlib import Path
from types import SimpleNamespace

from IPython.display import IFrame, display

from _bootstrap import ROOT
from utils import BaselineRoute, BaselineRouteGenerator, JeepneySystem, JeepneyRouteEnv

OUTPUT_DIR = Path(r"C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis") / "results" / "B4_baseline_routes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROUTES = 20
random_seed = secrets.randbits(32)
generator = BaselineRouteGenerator(
    min_area_m2=2_000_000,
    anchor_pool_size=96,
    max_attempts=500,
    seed=random_seed,
)

routes = generator.generate_routes(NUM_ROUTES, route_prefix="B4", seed=random_seed)
route_system = JeepneySystem.from_routes(routes)

summary = pd.DataFrame(
    [
        {
            "route_id": route.route_id,
            "anchors": list(route.ordered_anchor_node_ids),
            "area_m2": round(route.polygon_area_m2, 0),
            "length_m": round(route.path_length_m, 0),
            "attempt": route.attempt_index,
        }
        for route in routes
    ]
)
summary


,route_id,anchors,area_m2,length_m,attempt
0,B401,"[3141, 1090, 867, 872]",30515958.0,59278.0,1
1,B402,"[3, 102, 2724, 893]",30640836.0,82349.0,1
2,B403,"[2539, 1093, 619, 738]",20471483.0,52684.0,1
3,B404,"[1018, 1779, 473, 21]",18860790.0,54794.0,1
4,B405,"[109, 1093, 43, 2090]",52243781.0,47772.0,1
5,B406,"[998, 1427, 2611, 2224]",93553378.0,93846.0,1
6,B407,"[1193, 43, 2799, 1093]",6748731.0,25079.0,1
7,B408,"[70, 619, 2986, 3139]",128234589.0,76130.0,1
8,B409,"[43, 4, 1093, 1777]",6698986.0,28755.0,1
9,B410,"[1278, 3450, 843, 2101]",170511370.0,80702.0,1


## 2. RL Environment and Coordinate-Invariant Geometric Embeddings

Uses only the primal drivable network to build route geometry step by step, while encoding turn structure, sinuosity, and origin-relative features without absolute node IDs.

In [2]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)
env = JeepneyRouteEnv(
    drive_graph_raw=generator.drive_graph_raw,
    drive_graph_proj=generator.drive_graph_proj,
    seed=random_seed,
    max_steps=12,
)

def print_observation(obs: dict[str, np.ndarray]) -> None:
    print('shape:', obs['shape'])
    print('history:', obs['history'])
    print('topology:', obs['topology'])
    print('demand:', obs['demand'])
    print('global:', obs['global'])
    print('candidates:')
    for candidate_index, row in enumerate(obs['candidates']):
        print(f'  {candidate_index}: {row}')
    print('mask:', obs['action_mask'])

obs, info = env.reset()
print('reset state vector length:', info['state_vector'].shape[0])
print_observation(obs)

rng = np.random.default_rng(random_seed)
for step_index in range(5):
    valid_actions = np.flatnonzero(obs['action_mask'][:-1])
    action = int(rng.choice(valid_actions)) if len(valid_actions) else env.max_candidates
    obs, reward, terminated, truncated, info = env.step(action)
    print(f'step {step_index + 1}: action={action}, reward={reward:.3f}, terminated={terminated}, truncated={truncated}')
    print('turn angle rad:', info['turn_angle_rad'])
    print('sinuosity index:', info['sinuosity_index'])
    print('distance to origin m:', info['distance_to_origin_m'])
    print('bearing to origin rad:', info['bearing_to_origin_rad'])
    print('route area m2:', info['route_area_m2'])
    print('state vector:', info['state_vector'])
    print_observation(obs)
    if terminated or truncated:
        break


reset state vector length: 91
shape: [0. 0. 1. 0. 0. 0. 0.]
history: [0. 0. 0. 0. 0. 0. 0. 0.]
topology: [0.667 0.667 0.    0.5   0.5  ]
demand: [0.173 0.213 0.253 0.08 ]
global: [0.    0.    1.    0.    0.    0.    0.    0.    0.    0.    0.    0.
 0.    0.    0.    0.667 0.667 0.    0.5   0.5   0.173 0.213 0.253 0.08 ]
candidates:
  0: [-0.388 -0.922  0.007  0.5    0.     0.253  0.08   0.     1.     0.   ]
  1: [-0.969  0.245  0.005  0.5    0.     0.156 -0.017  0.     0.     0.   ]
  2: [-0.02   1.     0.022  0.5    0.     0.232  0.059  0.     0.     0.   ]
  3: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  4: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  5: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
mask: [1. 1. 1. 0. 0. 0. 1.]
step 1: action=1, reward=-0.000, terminated=False, truncated=False
turn angle rad: 0.0
sinuosity index: 1.0
distance to origin m: 32.28026746395677
bearing to origin rad: 3.141592653589793
route area m2: 0.0
state vector: [ 0.     0.    -1.     0.     0.     0.     0.     0.     0.     0.
  0.

In [3]:
route_edges = pd.DataFrame(
    [(int(u), int(v), "start_walk") for u, v in generator.drive_graph_raw.edges()],
    columns=["u", "v", "edge_type"],
)
route_manager = SimpleNamespace(
    edges=route_edges,
    _node_coords={
        int(row.base_node_id): (float(row.lat), float(row.lon))
        for row in generator.node_table.itertuples(index=False)
    },
)

def build_route_encoding(route_id: str) -> str:
    return f"route_id: {route_id}"

def interpret_embeddings(route: BaselineRoute) -> str:
    """Generate human-readable interpretations of route geometry."""
    route_path = route.path_node_ids
    if not route_path or len(route_path) < 2:
        return "Route too short to analyze."
    
    lines = []
    coords = route.path_latlon
    
    # Interpret shape: based on route area and length
    area = route.polygon_area_m2
    length = route.path_length_m
    area_to_length = area / max(length, 1.0)
    
    if area_to_length > 1000:
        lines.append("shape: Compact, efficient use of space")
    elif area_to_length > 200:
        lines.append("shape: Moderate coverage relative to distance")
    else:
        lines.append("shape: Linear route with extended length")
    
    # Interpret topology: based on node count (connectivity proxy)
    node_count = len(route_path)
    if node_count > 50:
        lines.append("topology: High node density, complex route structure")
    elif node_count > 25:
        lines.append("topology: Moderate complexity with multiple waypoints")
    else:
        lines.append("topology: Simple, direct route path")
    
    # Interpret demand: based on area size (proxy for coverage)
    if area > 50_000_000:
        lines.append("demand: Large service area, potentially high demand coverage")
    elif area > 10_000_000:
        lines.append("demand: Medium service area with moderate demand potential")
    else:
        lines.append("demand: Compact service area with focused demand")
    
    # Interpret history/shape: based on sinuosity (turn consistency)
    # Approximate sinuosity by comparing straight-line distance vs actual path length
    if len(coords) >= 2:
        import math
        start = coords[0]
        end = coords[-1]
        straight_dist = math.sqrt((end[0]-start[0])**2 + (end[1]-start[1])**2) * 111000  # rough m/degree
        actual_path = length
        sinuosity = actual_path / max(straight_dist, 1.0)
        
        if sinuosity > 1.5:
            lines.append("history: Winding route with many turns")
        elif sinuosity > 1.1:
            lines.append("history: Moderate turning pattern")
        else:
            lines.append("history: Direct path with minimal deviations")
    
    return chr(10).join(lines) if lines else "No embedding data available."

def build_route_notes(summary_df: pd.DataFrame, routes: list) -> dict[str, dict[str, str]]:
    notes: dict[str, dict[str, str]] = {}
    route_by_id = {r.route_id: r for r in routes}
    
    for row in summary_df.itertuples(index=False):
        route = route_by_id.get(row.route_id)
        notes[row.route_id] = {
            "encoding": build_route_encoding(row.route_id),
            "interpretation": interpret_embeddings(route) if route else "Route data unavailable.",
        }
    return notes

route_notes = build_route_notes(summary, routes)

html_path = route_system.export_route_toggle_html(
    route_manager,
    OUTPUT_DIR / "B4_route_explorer.html",
    title=f"B4 Jeepney Routes ({NUM_ROUTES})",
    route_notes=route_notes,
)
print(html_path)
display(IFrame(html_path.as_uri(), width=1200, height=900))


NameError: name 'travel_graph_manager' is not defined